In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/clustered_reviews.csv")

print("Dataset shape:", df.shape)

df[["reviews.text", "reviews.rating", "sentiment", "cluster", "cluster_name"]].head()

Dataset shape: (34615, 21)


C:\Users\d_fon\AppData\Local\Temp\ipykernel_1880\2669182736.py:3: DtypeWarning: Columns (1: name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/clustered_reviews.csv")


,reviews.text,reviews.rating,sentiment,cluster,cluster_name
0,This product so far has not disappointed. My c...,5.0,positive,1,General & Mixed Feedback
1,great for beginner or experienced person. Boug...,5.0,positive,3,"Kids, Family & Gifts"
2,Inexpensive tablet for him to use and learn on...,5.0,positive,5,Tablets & Value
3,I've had my Fire HD 8 two weeks now and I love...,4.0,positive,5,Tablets & Value
4,I bought this for my grand daughter when she c...,5.0,positive,5,Tablets & Value


In [ ]:
samples = []

for cluster_id in sorted(df["cluster"].unique()):
    
    cluster_df = df[df["cluster"] == cluster_id]
    
    for sentiment in ["positive", "neutral", "negative"]:
        
        sentiment_df = cluster_df[cluster_df["sentiment"] == sentiment]
        
        n = min(5, len(sentiment_df))
        
        if n > 0:
            sample = sentiment_df.sample(n=n, random_state=42)
            
            samples.append(sample)

summary_sample = pd.concat(samples)

print("Total sampled reviews:", len(summary_sample))

pd.crosstab(summary_sample["cluster_name"], summary_sample["sentiment"])

Total sampled reviews: 90


sentiment,negative,neutral,positive
cluster_name,,,
Ease of Use & Setup,5,5,5
Fire TV & Streaming,5,5,5
General & Mixed Feedback,5,5,5
"Kids, Family & Gifts",5,5,5
Kindle & Reading,5,5,5
Tablets & Value,5,5,5


In [ ]:
cluster_name = "Kindle & Reading"

cluster_reviews = summary_sample[summary_sample["cluster_name"] == cluster_name]

for sentiment in ["positive", "neutral", "negative"]:
    
    print(f"\n--- {sentiment.upper()} REVIEWS ---")
    
    reviews = cluster_reviews[cluster_reviews["sentiment"] == sentiment]["reviews.text"]
    
    for i, review in enumerate(reviews, 1):
        print(f"{i}. {review}")


--- POSITIVE REVIEWS ---
1. My husband and I both use this and really love it. We got kindle unlimited for the books but it's also great for playing games and surfing the web too.
2. Better than older kindles. Reading a book on a kindle is easier than holding a book.
3. I am more interested in a reader not a tablet the one thing I don't like about this reader is I cannot make the font when reading large enough
4. I bought this Kindle Fire for my daughter for Christmas, and she has completely enjoyed it. Build quality seems good, and I have been impressed with how Amazon has increased the functionality of its basic Kindle Fire over the iterations of this product (I also own a first generation Fire). It remains a very viable product and a joy to use.
5. I love the kindle. My kids love playing their Disney movies on it. They also have a ton of games we downloaded for free. I highly recommend!

--- NEUTRAL REVIEWS ---
1. I love it for reading but, it has issues when searching the Internet

In [ ]:
cluster_name = "Kindle & Reading"

cluster_data = df[df["cluster_name"] == cluster_name]
sample_data = summary_sample[
    summary_sample["cluster_name"] == cluster_name
]

total_reviews = len(cluster_data)
avg_rating = cluster_data["reviews.rating"].mean()

sentiment_distribution = (
    cluster_data["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
)

reviews_text = ""

for sentiment in ["positive", "neutral", "negative"]:

    reviews_text += f"\n{sentiment.upper()} REVIEWS:\n"

    reviews = sample_data[
        sample_data["sentiment"] == sentiment
    ]["reviews.text"]

    for review in reviews:
        reviews_text += f"- {review}\n"

prompt = f"""
You are analyzing customer reviews for the theme: {cluster_name}.

REAL CLUSTER STATISTICS:
- Total reviews: {total_reviews}
- Average rating: {avg_rating:.2f}
- Positive reviews: {sentiment_distribution.get('positive', 0)}%
- Neutral reviews: {sentiment_distribution.get('neutral', 0)}%
- Negative reviews: {sentiment_distribution.get('negative', 0)}%

The reviews below are a deliberately balanced qualitative sample.
Do not infer sentiment proportions from the sample.
Use the real statistics above when discussing overall customer satisfaction.

{reviews_text}

Generate a concise customer insight report with:

1. Overall summary
2. Main strengths mentioned by customers
3. Main complaints or pain points
4. Main customer use cases
5. One actionable business insight

Base the analysis only on the information provided.
If a review appears inconsistent with its sentiment label, prioritize the actual review text.
"""

print(prompt)


You are analyzing customer reviews for the theme: Kindle & Reading.

REAL CLUSTER STATISTICS:
- Total reviews: 4714
- Average rating: 4.67
- Positive reviews: 95.1%
- Neutral reviews: 3.0%
- Negative reviews: 1.9%

The reviews below are a deliberately balanced qualitative sample.
Do not infer sentiment proportions from the sample.
Use the real statistics above when discussing overall customer satisfaction.


POSITIVE REVIEWS:
- My husband and I both use this and really love it. We got kindle unlimited for the books but it's also great for playing games and surfing the web too.
- Better than older kindles. Reading a book on a kindle is easier than holding a book.
- I am more interested in a reader not a tablet the one thing I don't like about this reader is I cannot make the font when reading large enough
- I bought this Kindle Fire for my daughter for Christmas, and she has completely enjoyed it. Build quality seems good, and I have been impressed with how Amazon has increased the fun

In [ ]:
import importlib.util

packages = ["transformers", "torch"]

for package in packages:
    print(package, "->", "installed" if importlib.util.find_spec(package) else "not installed")

transformers -> installed
torch -> installed


In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.14.1+cpu
Transformers: 5.18.0
CUDA available: False


In [ ]:
import sys

print(sys.executable)

c:\Users\d_fon\ironhack-env\Scripts\python.exe


In [ ]:
import sys

!{sys.executable} -m pip --version

pip 26.2.1 from c:\Users\d_fon\ironhack-env\Lib\site-packages\pip (python 3.13)



In [ ]:
import sys

!{sys.executable} -m pip install torch transformers sentencepiece

In [ ]:
import sys
import torch
import transformers

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: c:\Users\d_fon\ironhack-env\Scripts\python.exe
PyTorch: 2.14.1+cpu
Transformers: 5.18.0
CUDA available: False


In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="google-t5/t5-small", device=-1)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

c:\Users\d_fon\ironhack-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\d_fon\.cache\huggingface\hub\models--google-t5--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("T5 model loaded successfully!")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 model loaded successfully!


In [ ]:
test_text = """
summarize: Customers like the Kindle because it is lightweight,
easy to use and convenient for reading at night.
Some customers complain about advertisements,
limited app availability and occasional software problems.
"""

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    min_new_tokens=15,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(summary)

the Kindle is lightweight, easy to use and convenient for reading at night. some customers complain about advertisements, limited app availability and occasional software problems.


In [ ]:
cluster_name = "Kindle & Reading"

kindle_sample = summary_sample[summary_sample["cluster_name"] == cluster_name]

kindle_text = " ".join(kindle_sample["reviews.text"].astype(str))

tokens = tokenizer(kindle_text, truncation=False)["input_ids"]

print("Reviews:", len(kindle_sample))
print("Tokens:", len(tokens))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (752 > 512). Running this sequence through the model will result in indexing errors


Reviews: 15
Tokens: 752


In [ ]:
def create_review_chunks(reviews, tokenizer, max_tokens=450):
    chunks = []
    current_chunk = []
    current_tokens = 0

    for review in reviews:
        review = str(review)
        review_tokens = len(tokenizer(review, truncation=False)["input_ids"])

        if current_tokens + review_tokens > max_tokens and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = []
            current_tokens = 0

        current_chunk.append(review)
        current_tokens += review_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks


kindle_chunks = create_review_chunks(kindle_sample["reviews.text"], tokenizer)

print("Number of chunks:", len(kindle_chunks))

for i, chunk in enumerate(kindle_chunks):
    token_count = len(tokenizer(chunk, truncation=False)["input_ids"])
    
    print(f"Chunk {i+1}: {token_count} tokens")

Number of chunks: 2
Chunk 1: 439 tokens
Chunk 2: 314 tokens


In [ ]:
chunk_summaries = []

for i, chunk in enumerate(kindle_chunks):

    input_text = "summarize: " + chunk

    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

    outputs = model.generate(**inputs, max_new_tokens=100, min_new_tokens=30, num_beams=4, early_stopping=True)

    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

    chunk_summaries.append(summary)

    print(f"\n--- SUMMARY CHUNK {i+1} ---")
    print(summary)


--- SUMMARY CHUNK 1 ---
my husband and I both use this kindle and really love it. it's great for playing games and surfing the web too. I bought this Kindle Fire for my daughter for christmas.

--- SUMMARY CHUNK 2 ---
this software downloads new purchases with a picture of the cover of the book on my homepage. the new books should be downloaded to my Library in List Form as in the original Kindle. this system disallows you from manipulating books the way you want.


In [ ]:
kindle_sample_check = kindle_sample[
    ["sentiment", "reviews.rating", "reviews.text"]
].copy()

kindle_sample_check["word_count"] = (
    kindle_sample_check["reviews.text"]
    .astype(str)
    .str.split()
    .str.len()
)

kindle_sample_check = kindle_sample_check.sort_values(
    ["sentiment", "word_count"],
    ascending=[True, False]
)

for _, row in kindle_sample_check.iterrows():
    print(
        f"\n[{row['sentiment'].upper()}]"
        f" Rating: {row['reviews.rating']}"
        f" | Words: {row['word_count']}"
    )
    print(row["reviews.text"])


[NEGATIVE] Rating: 2.0 | Words: 146
I am on my 3rd Kindle. This software downloads new purchases with a picture of the cover of the book on my homepage. Only about a fifth of the available space. So this would only allow you to buy at most four books at a time before the covers would be too small. The new books should be downloaded to my Library in List Form as in the original Kindle. In addition, once you have looked at a book in the homepage, it disappears and you have to search for it or place it in a category. Finally, this system disallows you from manipulating books the way you want. This OS is definitely NOT USER FRIENDLY. It requires a great deal of effort of the user. A lot of wasted data space and waste of time. I am requesting a return to the original Kindle OS.

[NEGATIVE] Rating: 2.0 | Words: 57
I have always gone to buy a kindle because I like the books and th games available. But I found them very unpredictable and unreliable. The color burns futon screen, then the scre

In [ ]:
structured_tokens = tokenizer(
    structured_reviews,
    truncation=False
)["input_ids"]

print("Structured input tokens:", len(structured_tokens))

Structured input tokens: 821


In [ ]:
positive_reviews = kindle_sample[
    kindle_sample["sentiment"] == "positive"
]["reviews.text"].tolist()

neutral_reviews = kindle_sample[
    kindle_sample["sentiment"] == "neutral"
]["reviews.text"].tolist()

negative_reviews = kindle_sample[
    kindle_sample["sentiment"] == "negative"
]["reviews.text"].tolist()


review_groups = []

for i in range(5):

    group = f"""
POSITIVE REVIEW:
{positive_reviews[i]}

NEUTRAL REVIEW:
{neutral_reviews[i]}

NEGATIVE REVIEW:
{negative_reviews[i]}
"""

    review_groups.append(group)


for i, group in enumerate(review_groups, 1):

    tokens = len(
        tokenizer(group, truncation=False)["input_ids"]
    )

    print(f"Group {i}: {tokens} tokens")

Group 1: 119 tokens
Group 2: 96 tokens
Group 3: 251 tokens
Group 4: 226 tokens
Group 5: 169 tokens


In [ ]:
balanced_chunks = []
current_chunk = ""
current_tokens = 0

for group in review_groups:

    group_tokens = len(
        tokenizer(group, truncation=False)["input_ids"]
    )

    if current_tokens + group_tokens > 430 and current_chunk:
        balanced_chunks.append(current_chunk)

        current_chunk = ""
        current_tokens = 0

    current_chunk += group
    current_tokens += group_tokens

if current_chunk:
    balanced_chunks.append(current_chunk)


print("Number of balanced chunks:", len(balanced_chunks))

for i, chunk in enumerate(balanced_chunks, 1):

    tokens = len(
        tokenizer(chunk, truncation=False)["input_ids"]
    )

    print(f"Chunk {i}: {tokens} tokens")

Number of balanced chunks: 3
Chunk 1: 214 tokens
Chunk 2: 251 tokens
Chunk 3: 394 tokens


In [ ]:
optimized_summaries = []

for i, chunk in enumerate(balanced_chunks):

    prompt = f"""
summarize: Analyze these Kindle customer reviews.
Identify the main benefits, common complaints, and customer use cases.
Write a concise and factual summary based only on the reviews.

{chunk}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        min_new_tokens=40,
        num_beams=5,
        no_repeat_ngram_size=3,
        length_penalty=1.1,
        early_stopping=True
    )

    summary = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    optimized_summaries.append(summary)

    print(f"\n--- OPTIMIZED SUMMARY {i+1} ---")
    print(summary)


--- OPTIMIZED SUMMARY 1 ---
POSITIVE REVIEW: I love it for reading but, it has issues when searching the internet at times. it seems the tabelt is designed to make one buy books from amazon or to buy apps.

--- OPTIMIZED SUMMARY 2 ---
easy to download books NEGATIVE REVIEW: I am on my 3rd Kindle. this software downloads new purchases with a picture of the cover on my homepage. once you have looked at a book in the homepage, it disappears.

--- OPTIMIZED SUMMARY 3 ---
POSITIVE REVIEW: I wanted a Kindle with a light feature for reading in bed at night. my kids love playing their Disney movies on it and they also have a ton of games we downloaded for free.


In [ ]:
combined_optimized = "\n".join(optimized_summaries)

final_prompt = f"""
summarize: Combine these customer review summaries into one concise overview.
Focus on the main benefits, complaints and customer use cases.
Do not add information that is not present in the summaries.

{combined_optimized}
"""

inputs = tokenizer(
    final_prompt,
    return_tensors="pt",
    max_length=512,
    truncation=True
)

outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    min_new_tokens=40,
    num_beams=5,
    no_repeat_ngram_size=3,
    length_penalty=1.1,
    early_stopping=True
)

t5_final_summary = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("--- T5-SMALL FINAL SUMMARY ---")
print(t5_final_summary)

--- T5-SMALL FINAL SUMMARY ---
POSITIVE REVIEW: I wanted a Kindle with a light feature for reading in bed at night. my kids love playing their Disney movies on it and they also have a ton of games we downloaded for free.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

flan_model_name = "google/flan-t5-small"

flan_tokenizer = AutoTokenizer.from_pretrained(
    flan_model_name
)

flan_model = AutoModelForSeq2SeqLM.from_pretrained(
    flan_model_name
)

print("FLAN-T5-small loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

c:\Users\d_fon\ironhack-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\d_fon\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5-small loaded successfully!


In [ ]:
flan_summaries = []

for i, chunk in enumerate(balanced_chunks):

    prompt = f"""
Analyze these Kindle customer reviews.

Identify:
- main benefits
- common complaints
- customer use cases

Write a concise factual summary combining the different customer opinions.
Do not copy individual reviews.
Do not add information that is not present in the reviews.

{chunk}
"""

    inputs = flan_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )

    outputs = flan_model.generate(
        **inputs,
        max_new_tokens=120,
        min_new_tokens=40,
        num_beams=5,
        no_repeat_ngram_size=3,
        length_penalty=1.1,
        early_stopping=True
    )

    summary = flan_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    flan_summaries.append(summary)

    print(f"\n--- FLAN SUMMARY {i+1} ---")
    print(summary)


--- FLAN SUMMARY 1 ---
POSITIVE REVIEW: I have bought at least 6 people this kindle it is perfect... i love it for the books but, it has issues when searching the Internet at times.

--- FLAN SUMMARY 2 ---
NEGATIVE REVIEW: I am on my 3rd Kindle. This software downloads new purchases with a picture of the cover of the book on my homepage. Only about a fifth of the available space. So this would only allow you to buy at most four books at a time before the covers would be too small. The new books should be downloaded to my Library in List Form as in the original Kindle

--- FLAN SUMMARY 3 ---
POSITIVE REVIEW: I bought this Kindle Fire for my daughter for Christmas, and she has completely enjoyed it. Build quality seems good, and I have been impressed with how Amazon has increased the functionality of its basic Kindle fire over the iterations of this product (I also own a first generation Fire)


In [ ]:
model_results = {
    "T5-small": t5_final_summary,
    "FLAN-T5-small": flan_summaries
}

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

bart_model_name = "sshleifer/distilbart-cnn-12-6"

bart_tokenizer = AutoTokenizer.from_pretrained(
    bart_model_name
)

bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    bart_model_name
)

print("DistilBART loaded successfully!")

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

c:\Users\d_fon\ironhack-env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\d_fon\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-12-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

DistilBART loaded successfully!


In [ ]:
bart_summaries = []

for i, chunk in enumerate(balanced_chunks):

    inputs = bart_tokenizer(
        chunk,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    outputs = bart_model.generate(
        **inputs,
        max_new_tokens=120,
        min_new_tokens=40,
        num_beams=5,
        no_repeat_ngram_size=3,
        length_penalty=1.1,
        early_stopping=True
    )

    summary = bart_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    bart_summaries.append(summary)

    print(f"\n--- BART SUMMARY {i+1} ---")
    print(summary)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BART SUMMARY 1 ---
 The kindle has unlimited unlimited for the books but it's also great for playing games and surfing the web too . It has issues when searching the Internet at times . It seems the tabelt is designed to make one buy books from amazon .


[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BART SUMMARY 2 ---
 This software downloads new purchases with a picture of the cover of the book on my homepage . Only about a fifth of the available space . This OS is definitely NOT USER FRIENDLY. It requires a great deal of effort of the user .


[transformers] Both `max_new_tokens` (=120) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=40) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- BART SUMMARY 3 ---
 The Kindle Fire remains a very viable product and a joy to use . Amazon should include this charger with the Kindle Fire . The color burns futon screen, then the screens starts jumping . The light feature is light for reading in bed at night .


In [ ]:
combined_bart = " ".join(bart_summaries)

inputs = bart_tokenizer(
    combined_bart,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)

outputs = bart_model.generate(
    **inputs,
    max_length=160,
    min_length=60,
    num_beams=5,
    no_repeat_ngram_size=3,
    length_penalty=1.1,
    early_stopping=True
)

bart_final_summary = bart_tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("--- DISTILBART FINAL SUMMARY ---")
print(bart_final_summary)

--- DISTILBART FINAL SUMMARY ---
 The kindle has unlimited unlimited unlimited for the books but it's also great for playing games and surfing the web too . It has issues when searching the Internet at times . It seems the tabelt is designed to make one buy books from amazon . The light feature is light for reading in bed at night .


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
import os

print("Current directory:", os.getcwd())
print("Datasets exists:", os.path.exists("../datasets/clustered_reviews.csv"))

Current directory: /content
Datasets exists: False


In [4]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
import os
import pandas as pd

path = "/content/drive/MyDrive/ironhack/clustered_reviews.csv"

print("File exists:", os.path.exists(path))

df = pd.read_csv(path)

print("Dataset shape:", df.shape)
print(df["cluster_name"].value_counts())

File exists: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ironhack/clustered_reviews.csv'

In [6]:
import os

for root, dirs, files in os.walk("/content/drive"):
    if "clustered_reviews.csv" in files:
        print(os.path.join(root, "clustered_reviews.csv"))

/content/drive/MyDrive/MyDrive/ironhack/clustered_reviews.csv


In [7]:
import pandas as pd

path = "/content/drive/MyDrive/MyDrive/ironhack/clustered_reviews.csv"

df = pd.read_csv(path)

print("Dataset shape:", df.shape)
print()
print(df["cluster_name"].value_counts())

Dataset shape: (34615, 21)

cluster_name
General & Mixed Feedback    17011
Tablets & Value              5426
Kindle & Reading             4714
Kids, Family & Gifts         2765
Ease of Use & Setup          2761
Fire TV & Streaming          1938
Name: count, dtype: int64


/tmp/ipykernel_2447/2189594103.py:5: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


In [8]:
cluster_name = "Kindle & Reading"

kindle_data = df[
    df["cluster_name"] == cluster_name
]

samples = []

for sentiment in ["positive", "neutral", "negative"]:

    sentiment_df = kindle_data[
        kindle_data["sentiment"] == sentiment
    ]

    sample = sentiment_df.sample(
        n=5,
        random_state=42
    )

    samples.append(sample)

kindle_sample = pd.concat(samples)

print("Total reviews:", len(kindle_sample))
print()
print(kindle_sample["sentiment"].value_counts())

Total reviews: 15

sentiment
positive    5
neutral     5
negative    5
Name: count, dtype: int64


In [9]:
!pip install -q -U transformers accelerate sentencepiece

In [10]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.18.0
Accelerate: 1.15.0


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto"
)

print("Model loaded!")
print("Device:", next(model.parameters()).device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!
Device: cuda:0


In [12]:
# Real statistics from the full Kindle cluster
kindle_full = df[df["cluster_name"] == "Kindle & Reading"]

total_reviews = len(kindle_full)
avg_rating = kindle_full["reviews.rating"].mean()

sentiment_pct = (
    kindle_full["sentiment"]
    .value_counts(normalize=True)
    .mul(100)
)

print("Total reviews:", total_reviews)
print("Average rating:", round(avg_rating, 2))
print(sentiment_pct.round(1))

Total reviews: 4714
Average rating: 4.67
sentiment
positive    95.1
neutral      3.0
negative     1.9
Name: proportion, dtype: float64


In [13]:
reviews_text = ""

for i, (_, row) in enumerate(kindle_sample.iterrows(), 1):
    reviews_text += (
        f"\nReview {i} "
        f"(Rating: {row['reviews.rating']}/5):\n"
        f"{row['reviews.text']}\n"
    )

print(reviews_text[:2000])


Review 1 (Rating: 5.0/5):
My husband and I both use this and really love it. We got kindle unlimited for the books but it's also great for playing games and surfing the web too.

Review 2 (Rating: 5.0/5):
Better than older kindles. Reading a book on a kindle is easier than holding a book.

Review 3 (Rating: 4.0/5):
I am more interested in a reader not a tablet the one thing I don't like about this reader is I cannot make the font when reading large enough

Review 4 (Rating: 5.0/5):
I bought this Kindle Fire for my daughter for Christmas, and she has completely enjoyed it. Build quality seems good, and I have been impressed with how Amazon has increased the functionality of its basic Kindle Fire over the iterations of this product (I also own a first generation Fire). It remains a very viable product and a joy to use.

Review 5 (Rating: 5.0/5):
I love the kindle. My kids love playing their Disney movies on it. They also have a ton of games we downloaded for free. I highly recommend!

R

In [16]:
prompt_v3 = f"""
You are a customer review analyst.

Analyze customer feedback for the category: "Kindle & Reading".

REAL CLUSTER STATISTICS:
- Total reviews: {total_reviews}
- Average rating: {avg_rating:.2f}/5
- Positive sentiment: {sentiment_pct.get('positive', 0):.1f}%
- Neutral sentiment: {sentiment_pct.get('neutral', 0):.1f}%
- Negative sentiment: {sentiment_pct.get('negative', 0):.1f}%

SAMPLING NOTE:
The 15 reviews below were deliberately balanced:
5 positive, 5 neutral and 5 negative.

This sample is designed to expose strengths and problems.
It does NOT represent the real sentiment distribution.
Use the real statistics above for overall customer sentiment.

STRICT GROUNDING RULES:
- Use ONLY information explicitly present in the reviews or statistics.
- Do NOT infer additional product problems or features.
- Do NOT call something "common", "frequent" or "widespread" based on this small sample.
- Distinguish observations from individual reviews from overall cluster statistics.
- If only one review mentions an issue, present it as an example, not a general pattern.
- Do NOT invent information about customer support, pricing, storage, compatibility or features.
- Star ratings and text may disagree; prioritize the review text.
- Synthesize the evidence rather than copying sentences.

TASK:
Write a concise consumer insight article of 180-230 words.

Use exactly these sections:

### Overall Customer Perception
Use the real cluster statistics to describe overall satisfaction.

### Strengths & Use Cases
Describe strengths and use cases supported by the review sample.

### Complaints & Trade-offs
Describe only limitations explicitly supported by the reviews.

### Buyer Takeaway
Provide a balanced conclusion based only on the evidence.

REVIEWS:
{reviews_text}
"""

messages_v3 = [
    {
        "role": "system",
        "content": (
            "You are an evidence-grounded customer review analyst. "
            "Never add facts that are not explicitly supported by the supplied data."
        )
    },
    {
        "role": "user",
        "content": prompt_v3
    }
]

formatted_prompt_v3 = tokenizer.apply_chat_template(
    messages_v3,
    tokenize=False,
    add_generation_prompt=True
)

inputs_v3 = tokenizer(
    formatted_prompt_v3,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs_v3 = model.generate(
        **inputs_v3,
        max_new_tokens=450,
        do_sample=False,
        repetition_penalty=1.08
    )

generated_tokens_v3 = outputs_v3[:, inputs_v3["input_ids"].shape[1]:]

qwen_summary_v3 = tokenizer.decode(
    generated_tokens_v3[0],
    skip_special_tokens=True
)

print(qwen_summary_v3)

### Overall Customer Perception
Overall, customers express high levels of satisfaction with the Kindle devices, indicated by an average rating of 4.67/5 and a positive sentiment of 95.1%. Despite some minor complaints, the majority of reviews highlight the device’s ease of use for reading and entertainment purposes.

### Strengths & Use Cases
Customers appreciate the Kindle's versatility beyond just reading, noting its suitability for gaming and browsing the internet. Reviews emphasize the comfort of using the device while reading, praising its build quality and the variety of content available. Parents particularly value the Kindle for children, enjoying the ability to play Disney movies and access educational games.

### Complaints & Trade-offs
While the device generally performs well, there are notable drawbacks highlighted by specific reviews. Some users find the interface confusing or difficult to navigate, especially regarding ad pop-ups and limited customization options. Technic

In [19]:
def generate_cluster_summary(cluster_name, df, model, tokenizer):

    # 1. Select full cluster
    cluster_df = df[df["cluster_name"] == cluster_name].copy()

    # 2. Real cluster statistics
    total_reviews = len(cluster_df)
    avg_rating = cluster_df["reviews.rating"].mean()

    sentiment_pct = (
        cluster_df["sentiment"]
        .value_counts(normalize=True)
        .mul(100)
    )

    # 3. Balanced sample: 5 positive + 5 neutral + 5 negative
    samples = []

    for sentiment in ["positive", "neutral", "negative"]:
        sentiment_df = cluster_df[
            cluster_df["sentiment"] == sentiment
        ]

        n = min(5, len(sentiment_df))

        if n > 0:
            samples.append(
                sentiment_df.sample(
                    n=n,
                    random_state=42
                )
            )

    sample_df = pd.concat(samples)

    # 4. Convert reviews into text
    reviews_text = ""

    for i, (_, row) in enumerate(sample_df.iterrows(), 1):
        reviews_text += (
            f"\nReview {i} "
            f"(Rating: {row['reviews.rating']}/5):\n"
            f"{row['reviews.text']}\n"
        )

    # 5. Grounded prompt
    prompt = f"""
You are a customer review analyst.

Analyze customer feedback for the category: "{cluster_name}".

REAL CLUSTER STATISTICS:
- Total reviews: {total_reviews}
- Average rating: {avg_rating:.2f}/5
- Positive sentiment: {sentiment_pct.get('positive', 0):.1f}%
- Neutral sentiment: {sentiment_pct.get('neutral', 0):.1f}%
- Negative sentiment: {sentiment_pct.get('negative', 0):.1f}%

SAMPLING NOTE:
The reviews below were deliberately balanced across positive,
neutral and negative sentiment whenever enough reviews were available.

This sample is designed to expose strengths and problems.
It does NOT represent the real sentiment distribution.
Use the real statistics above for overall customer sentiment.

STRICT GROUNDING RULES:
- Use ONLY information explicitly present in the reviews or statistics.
- Do NOT infer additional product problems or features.
- Do NOT call something "common", "frequent" or "widespread"
  based on this small sample.
- If only one review mentions an issue, present it as an example,
  not as a general pattern.
- Do NOT invent information about customer support, pricing,
  storage, compatibility or features.
- Star ratings and text may disagree; prioritize the review text.
- Synthesize the evidence rather than copying review sentences.

TASK:
Write a concise consumer insight article of 180-230 words.

Use exactly these sections:

### Overall Customer Perception
Use the real cluster statistics to describe overall satisfaction.

### Strengths & Use Cases
Describe strengths and use cases supported by the review sample.

### Complaints & Trade-offs
Describe only limitations explicitly supported by the reviews.

### Buyer Takeaway
Provide a balanced conclusion based only on the evidence.

REVIEWS:
{reviews_text}
"""

    # 6. Chat format
    messages = [
        {
            "role": "system",
            "content": (
                "You are an evidence-grounded customer review analyst. "
                "Never add facts that are not explicitly supported "
                "by the supplied data."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    # 7. Generate summary
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=False,
            repetition_penalty=1.08
        )

    generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]

    summary = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

    return summary

In [20]:
test_summary = generate_cluster_summary(
    "Kindle & Reading",
    df,
    model,
    tokenizer
)

print(test_summary)

### Overall Customer Perception
Overall, customers express high levels of satisfaction with the Kindle devices, indicated by an average rating of 4.67/5 and a positive sentiment of 95.1%. Reviews highlight the ease of reading, variety of content options, and improved functionalities over previous models, suggesting that the product meets or exceeds expectations.

### Strengths & Use Cases
Customers appreciate the Kindle’s versatility beyond just reading, noting its suitability for gaming and browsing the internet. Reviewers emphasize the ease of using the device for nighttime reading and downloading various types of content, including e-books and Disney movies. Parents particularly value the Kindle’s ability to engage children with educational content and entertainment.

### Complaints & Trade-offs
Despite the widespread praise, some users report specific issues. For instance, one reviewer finds the font size adjustment challenging, while another notes that advertisements appear freque

In [21]:
cluster_summaries = {}

cluster_names = df["cluster_name"].unique()

for cluster_name in cluster_names:

    print(f"\nGenerating: {cluster_name}...")

    summary = generate_cluster_summary(
        cluster_name,
        df,
        model,
        tokenizer
    )

    cluster_summaries[cluster_name] = summary

    print("Done!")

print("\nAll cluster summaries generated!")


Generating: General & Mixed Feedback...
Done!

Generating: Kids, Family & Gifts...
Done!

Generating: Tablets & Value...
Done!

Generating: Kindle & Reading...
Done!

Generating: Ease of Use & Setup...
Done!

Generating: Fire TV & Streaming...
Done!

All cluster summaries generated!


In [22]:
summaries_df = pd.DataFrame(
    list(cluster_summaries.items()),
    columns=["cluster_name", "generated_summary"]
)

summaries_df

,cluster_name,generated_summary
0,General & Mixed Feedback,"### Overall Customer Perception\nOverall, cust..."
1,"Kids, Family & Gifts","### Overall Customer Perception\nOverall, cust..."
2,Tablets & Value,"### Overall Customer Perception\nOverall, cust..."
3,Kindle & Reading,"### Overall Customer Perception\nOverall, cust..."
4,Ease of Use & Setup,"### Overall Customer Perception\nOverall, cust..."
5,Fire TV & Streaming,"### Overall Customer Perception\nOverall, cust..."


In [23]:
output_path = "/content/drive/MyDrive/MyDrive/ironhack/category_summaries.csv"

summaries_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: /content/drive/MyDrive/MyDrive/ironhack/category_summaries.csv


In [24]:
import os

print("File exists:", os.path.exists(output_path))
print("Number of summaries:", len(summaries_df))

File exists: True
Number of summaries: 6


In [25]:
for cluster_name, summary in cluster_summaries.items():

    print("\n" + "=" * 80)
    print(cluster_name.upper())
    print("=" * 80)
    print(summary)


GENERAL & MIXED FEEDBACK
### Overall Customer Perception
Overall, customers exhibit high satisfaction with the product, indicated by an average rating of 4.54/5 and a positive sentiment of 92%. Despite a slight dip in neutral and negative sentiments (4.9% and 3.1%, respectively), the overwhelming majority of reviews express approval, suggesting broad customer acceptance.

### Strengths & Use Cases
Customers appreciate the product’s versatility and value. Reviewers highlight its ease of use, particularly in travel scenarios (e.g., Tablet Review 3) and its functionality as a smart home assistant (Echo Review 5). Additionally, the product is praised for its affordability and inclusion of necessary features such as built-in filters (Tablet Review 2).

### Complaints & Trade-offs
While most issues mentioned are minor, there are notable points of concern. Users report difficulties with certain functionalities, especially regarding app usage and custom ROM installation (Tablet Reviews 3 and 

# Task 3 — GenAI Review Summarization

## Objective

The goal of this task was to transform the review clusters created in Task 2 into short consumer insight articles.

Instead of summarizing individual reviews, the objective was to identify and synthesize:

- overall customer perception
- main strengths and use cases
- relevant complaints and trade-offs
- a useful takeaway for potential customers

The final system generates one article for each of the six clusters identified during the clustering stage.


## Review Sampling Strategy

The dataset is strongly dominated by positive reviews.

Because a random sample would contain mostly positive feedback, it could hide important complaints and limitations.

For qualitative summarization, we therefore created a balanced sample for each cluster:

- 5 positive reviews
- 5 neutral reviews
- 5 negative reviews

when enough reviews were available.

This balanced sample is NOT representative of the real sentiment distribution.

To avoid misleading the language model, the prompt also includes statistics calculated from the complete cluster:

- total number of reviews
- average rating
- real positive percentage
- real neutral percentage
- real negative percentage

This allows the model to use the balanced sample to discover strengths and problems while using the complete dataset statistics to describe overall customer sentiment.


## Model Experiments

Several pretrained Hugging Face models were evaluated using the same Kindle & Reading review sample.

### T5-small

T5-small was used as the first summarization baseline.

The model was able to compress individual reviews but struggled to combine multiple customer opinions into a coherent analysis.

Because of its limited context, reviews were divided into balanced chunks and summarized using a Map-Reduce strategy.

Main limitation:

> The output remained highly extractive and important information was lost during the final reduction step.


### FLAN-T5-small

FLAN-T5-small was tested to evaluate whether instruction tuning would improve the results.

More explicit instructions were provided to identify benefits, complaints and customer use cases.

Although instruction following improved slightly, the output remained largely extractive and did not consistently synthesize patterns across multiple reviews.


### DistilBART-CNN

DistilBART produced better multi-review summaries than T5-small and FLAN-T5-small.

It was able to combine more information from different reviews, but the final summaries still showed:

- repetition
- loss of relevant complaints
- excessive dependence on original review wording
- information loss during the Map-Reduce process

DistilBART was therefore considered the strongest traditional summarization baseline, but not sufficient for the final consumer insight generation.


## Final Model — Qwen2.5-3B-Instruct

To improve multi-review reasoning and instruction following, we tested:

**Qwen/Qwen2.5-3B-Instruct**

The model was executed on a Google Colab Tesla T4 GPU through Hugging Face Transformers.

Unlike the previous models, Qwen could process the complete 15-review sample in a single prompt.

This removed the need for Map-Reduce summarization and allowed the model to consider positive, neutral and negative feedback simultaneously.

Qwen produced substantially more coherent and structured consumer insight articles than the smaller summarization models.

The final model was therefore:

> **Qwen2.5-3B-Instruct**


## Prompt Engineering

Three prompt iterations were evaluated.

### Version 1

The initial prompt requested:

- overall customer perception
- strengths
- complaints
- use cases
- buyer takeaway

The output showed strong synthesis but occasionally generalized beyond the supplied evidence and was truncated by the generation limit.


### Version 2

The second prompt introduced:

- a fixed article structure
- stronger anti-hallucination instructions
- explicit treatment of conflicting reviews
- a larger generation limit

The structure improved significantly, but some unsupported generalizations were still observed.


### Version 3 — Final Prompt

The final prompt introduced stricter grounding rules.

The model was explicitly instructed to:

- use only information present in the supplied data
- avoid inventing product features or problems
- avoid describing isolated observations as common problems
- distinguish the balanced review sample from the real cluster statistics
- prioritize review text when star ratings and text disagree
- synthesize rather than copy individual reviews

This version was selected for the final pipeline.


## Final Pipeline

The complete summarization pipeline is:

    Cluster
       ↓
    Calculate real cluster statistics
       ↓
    Select balanced review sample
    (positive + neutral + negative)
       ↓
    Build grounded prompt
       ↓
    Qwen2.5-3B-Instruct
       ↓
    Consumer Insight Article

The process was implemented as a reusable function and automatically applied to all six clusters:

1. General & Mixed Feedback
2. Kids, Family & Gifts
3. Tablets & Value
4. Kindle & Reading
5. Ease of Use & Setup
6. Fire TV & Streaming

This produced six consumer insight articles using the same model, sampling strategy and prompt.


## Key Observations

The experiments showed a clear difference between traditional summarization models and an instruction-tuned LLM.

T5-small and FLAN-T5-small mainly compressed existing text.

DistilBART improved aggregation but still struggled to create a complete multi-review analysis.

Qwen2.5-3B-Instruct was substantially better at:

- combining information from multiple reviews
- following analytical instructions
- separating strengths from complaints
- incorporating real cluster statistics
- producing structured and readable consumer insights

Another important observation was that review ratings and review text are not always consistent.

For example, some reviews classified as negative from their star rating contained positive textual feedback.

For this reason, the final prompt explicitly instructs the LLM to prioritize the actual review text when interpreting individual feedback.


## Limitations

The final approach still has important limitations.

First, only a small balanced sample of reviews is provided to the LLM. This improves visibility of minority opinions but does not guarantee that every important topic in a cluster is represented.

Second, generative models can still produce unsupported generalizations even when strict grounding instructions are included.

Manual evaluation of the generated articles identified occasional statements that went beyond the supplied evidence.

Therefore:

> Prompt engineering can reduce hallucination risk, but it does not eliminate it.

The generated articles should therefore be interpreted as AI-assisted consumer insights rather than fully verified factual reports.

A production system could improve reliability through larger or more representative samples, automated evidence retrieval, citation of supporting reviews, or human validation.


## Final Conclusion

Among the evaluated models, **Qwen2.5-3B-Instruct produced the strongest multi-review synthesis**.

The main improvement was not simply better text generation. The model was able to transform multiple customer opinions and aggregate statistics into a structured consumer-oriented analysis.

The experiments also demonstrated an important limitation of Generative AI: fluent output is not necessarily fully grounded output.

For this reason, the final solution combines:

**real cluster statistics + balanced review sampling + prompt grounding + manual evaluation**

rather than relying exclusively on the generated text.

Task 3 successfully produced automated consumer insight articles for all six review clusters while also documenting the limitations and risks of LLM-based summarization.